# Colab repository and Drive setup

Run this notebook first. Select a GPU runtime before executing it. Authentication tokens are never stored in notebook source or output.

In [ ]:
import sys, subprocess
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda, 'available:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU:', p.name, 'memory GiB:', round(p.total_memory / 2**30, 2))
else:
    print('Select Runtime > Change runtime type > GPU before training.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GITHUB_USERNAME = "Harryphan72007"
GITHUB_REPOSITORY = "aerial-object-detection-benchmark"
DEFAULT_BRANCH = "main"

REPOSITORY_URL = (
    f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPOSITORY}.git"
)
LOCAL_REPOSITORY = f"/content/{GITHUB_REPOSITORY}"
DRIVE_ROOT = (
    "/content/drive/MyDrive/"
    "visdrone_architecture_benchmark"
)
assert GITHUB_USERNAME != "<MY_GITHUB_USERNAME>", 'Edit the username placeholder before cloning'

In [ ]:
import os, sys
if os.path.isdir(LOCAL_REPOSITORY) and not os.path.isdir(os.path.join(LOCAL_REPOSITORY, '.git')):
    raise RuntimeError(f'Existing non-Git directory: {LOCAL_REPOSITORY}')
if LOCAL_REPOSITORY not in sys.path:
    sys.path.insert(0, LOCAL_REPOSITORY)
if not os.path.isdir(os.path.join(LOCAL_REPOSITORY, '.git')):
    subprocess.run(['git', 'clone', '--branch', DEFAULT_BRANCH, REPOSITORY_URL, LOCAL_REPOSITORY], check=True)
sys.path.insert(0, LOCAL_REPOSITORY)
from src.colab_setup import (clone_or_update_repository, install_project,
    initialize_drive_directories, load_project_config,
    print_environment_summary, validate_drive_writable)
clone_or_update_repository(REPOSITORY_URL, LOCAL_REPOSITORY, DEFAULT_BRANCH)
os.chdir(LOCAL_REPOSITORY)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
install_project(LOCAL_REPOSITORY)
config = load_project_config('project_config.yaml')
print_environment_summary()

In [ ]:
paths = initialize_drive_directories(DRIVE_ROOT)
validate_drive_writable(DRIVE_ROOT)
required_paths = {
    'dataset_2class': paths.coco('2class'),
    'dataset_10class': paths.coco('10class'),
    'checkpoints': paths.checkpoints,
    'registry': paths.registry_dir,
    'reports': paths.reports,
    'result_bundles': paths.result_bundles,
}
for name, path in required_paths.items(): print(f'{name}: {path} [{path.exists()}]')

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_workflow_utils.py', 'tests/test_checkpoint_registry.py', 'tests/test_class_mapping.py'], check=True)

## Recommended order

`00_colab_repository_setup` → `00_environment_and_data_setup` → `01_dataset_analysis` → `02`–`06` individual training → `07_evaluate_all_models` → `08_architecture_visualization` → `09_error_analysis` → `10_generate_final_report` → `11_sync_results_to_github`.

Place VisDrone archives in `DRIVE_ROOT/datasets/raw/`, then run notebook 00's data setup notebook before training.